# Stage 4 — Transformer fine-tuning (MuRIL / XLM-R / IndicBERTv2)

Run this on **Google Colab with a GPU** runtime (Runtime → Change runtime type → GPU).

**Before you start (on your laptop):**
```bash
python src/package_for_colab.py      # creates colab_bundle.zip
```
Then upload `colab_bundle.zip` in Step 2 below.

**Flow:** check GPU → install → upload bundle → train all models → download `colab_outputs.zip`.
Back on your laptop: `python src/import_colab_outputs.py`.

## Step 1 — Check GPU and install dependencies

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU'
!pip install -q -U transformers accelerate sentencepiece scikit-learn pandas

## Step 2 — Upload and unpack the bundle

In [ ]:
import os, zipfile
from google.colab import files

print('Select colab_bundle.zip (made by src/package_for_colab.py) ...')
up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('.')
os.chdir('project')
print('cwd:', os.getcwd())
print('data files:', os.listdir('data/processed'))

## Step 3 — Train all models × variants

6 runs by default (3 models × 2 variants). Edit the config to run fewer.
Each run evaluates on the in-domain test set **and** the Punjabi seed, appending
to `results/metrics/comparison.csv`.

In [ ]:
import subprocess, sys

MODELS   = ['muril', 'xlmr', 'indicbert']   # drop any to save time
# 'rp' = trained on the Romanized Punjabi dataset (project core).
# 'full'/'hinglish' = Hinglish-trained baselines (for the transfer comparison).
VARIANTS = ['rp', 'full', 'hinglish']
EPOCHS   = 4

for m in MODELS:
    for v in VARIANTS:
        print('\n' + '='*70)
        print(f'TRAINING  model={m}  variant={v}')
        print('='*70)
        r = subprocess.run(
            [sys.executable, 'src/train_transformer.py',
             '--model', m, '--variant', v, '--epochs', str(EPOCHS)],
        )
        if r.returncode != 0:
            print(f'!! {m}/{v} failed (continuing)')
print('\nAll runs done.')

## Step 3b — Robustness evaluation (Stage 6)
Runs each trained model on perturbed test text (spelling noise, char elongation,
code-mix injection, etc.) and measures the macro-F1 drop. Uses the in-session
checkpoints — no re-download. Results (CSVs + figures) land in `results/` and come
back in the outputs zip.

In [ ]:
import subprocess, sys, os
for m in MODELS:
    for v in VARIANTS:
        ckpt = f'models/{m}_{v}'
        if not os.path.isdir(ckpt):
            print(f'skip {ckpt} (not trained)'); continue
        print(f'\nROBUSTNESS  {m}/{v}')
        subprocess.run([sys.executable, 'src/robustness_eval.py',
                        '--model-type', 'transformer', '--checkpoint', ckpt,
                        '--model-name', m, '--variant', v])
print('\nRobustness done.')

## Step 4 — Review results

In [ ]:
import pandas as pd
df = pd.read_csv('results/metrics/comparison.csv')
df.sort_values(['eval_on', 'macro_f1'], ascending=[True, False])

## Step 5 — Package outputs and download
Zips metrics (always) and the trained model checkpoints. Models are large — set
`INCLUDE_MODELS = False` if you only want the metrics.

In [ ]:
import zipfile, os
from google.colab import files

INCLUDE_MODELS = False
out = '/content/colab_outputs.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk('results'):
        for f in fs:
            p = os.path.join(root, f)
            z.write(p, arcname=f'project/{p}')
    if INCLUDE_MODELS:
        for root, _, fs in os.walk('models'):
            if '_trainer' in root:        # skip intermediate trainer checkpoints
                continue
            for f in fs:
                p = os.path.join(root, f)
                z.write(p, arcname=f'project/{p}')
print('wrote', out, round(os.path.getsize(out)/1e6, 1), 'MB')
files.download(out)

## Back on your laptop
```bash
# move the downloaded colab_outputs.zip into the project root, then:
python src/import_colab_outputs.py
# (or: python src/import_colab_outputs.py --zip ~/Downloads/colab_outputs.zip)
# add --no-models if you only want the metrics
```
Then continue with Stage 5 (comparison), Stage 6 (robustness), etc.